# 01 — Data Audit

**CareRisk**: predicting and optimizing intervention allocation for 30-day hospital readmission.

Dataset: [UCI Diabetes 130-US hospitals (1999-2008)](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008) — 101,766 encounters, 50 columns.

Goal of this notebook: understand every column before touching a model, and write down every cleaning decision with a reason. No modeling happens here.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 60)

df = pd.read_csv('../data/raw/diabetic_data.csv')
ids = pd.read_csv('../data/raw/IDS_mapping.csv', header=None)
df.shape

(101766, 50)

## 1. Column-by-column audit

Missingness in this dataset is encoded three ways: real `NaN`, the literal string `'?'`, and (for `max_glu_serum`/`A1Cresult`) `NaN` meaning "not tested" — which is itself informative, not missing-at-random.

In [2]:
audit = []
for c in df.columns:
    nq = int((df[c] == '?').sum()) if df[c].dtype == object else 0
    nn = int(df[c].isna().sum())
    audit.append({
        'column': c,
        'dtype': str(df[c].dtype),
        'nulls': nn,
        'question_marks': nq,
        'pct_missing': round(100 * (nn + nq) / len(df), 1),
        'nunique': df[c].nunique(),
    })
audit_df = pd.DataFrame(audit)
audit_df

,column,dtype,nulls,question_marks,pct_missing,nunique
0,encounter_id,int64,0,0,0.0,101766
1,patient_nbr,int64,0,0,0.0,71518
2,race,object,0,2273,2.2,6
3,gender,object,0,0,0.0,3
4,age,object,0,0,0.0,10
5,weight,object,0,98569,96.9,10
6,admission_type_id,int64,0,0,0.0,8
7,discharge_disposition_id,int64,0,0,0.0,26
8,admission_source_id,int64,0,0,0.0,17
9,time_in_hospital,int64,0,0,0.0,14


In [3]:
audit_df[audit_df['pct_missing'] > 0].sort_values('pct_missing', ascending=False)

,column,dtype,nulls,question_marks,pct_missing,nunique
5,weight,object,0,98569,96.9,10
22,max_glu_serum,object,96420,0,94.7,3
23,A1Cresult,object,84748,0,83.3,3
11,medical_specialty,object,0,49949,49.1,73
10,payer_code,object,0,40256,39.6,18
2,race,object,0,2273,2.2,6
20,diag_3,object,0,1423,1.4,790
19,diag_2,object,0,358,0.4,749


**Decisions:**

- `weight` (97% missing) → **drop column**. Almost no signal left after dropping rows would be worse than dropping the column.
- `payer_code` (40%), `medical_specialty` (49%) → **keep**, recode `?` to explicit `"Unknown"` category rather than imputing or dropping. Missingness on `medical_specialty` in particular likely correlates with which department saw the patient, which is plausibly predictive.
- `race` (2.2%) → **keep**, recode `?` to `"Unknown"`. Too few rows to drop, imputing demographic data is not defensible.
- `diag_1/2/3` (~0-1.4% `?`) → **keep**, recode `?` to `"Unknown"`, then group ICD-9 codes into clinical categories (done in feature engineering, not here) rather than using 700+ raw codes directly.
- `max_glu_serum`, `A1Cresult` → `NaN` means "not tested", **not missing data**. Recoded to an explicit `"Not tested"` level, never imputed.

In [4]:
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
constant_cols

['examide', 'citoglipton']

`examide` and `citoglipton` take a single value across all 101,766 encounters → **drop**, zero information.

## 2. Target variable

In [5]:
df['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [6]:
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)
print('Positive rate:', round(100 * df['readmitted_30d'].mean(), 2), '%')

Positive rate: 11.16 %


Collapsing `<30` / `>30` / `NO` to binary `readmitted_30d`. Positive rate ≈ 11.2% — **imbalanced**. Accuracy will not be a usable headline metric later; ROC-AUC/PR-AUC/recall-at-capacity are.

## 3. Leakage check — discharge disposition (death/hospice)

A patient who died during the encounter cannot be readmitted. If those rows stay in with `readmitted_30d = 0`, the model is rewarded for learning "predict death → predict no readmission", which is not a decision-relevant signal for who should get a care-management intervention.

In [7]:
disp_map = ids.iloc[11:40].copy()
disp_map.columns = ['id_str', 'description']
disp_map = disp_map[disp_map['id_str'] != 'discharge_disposition_id']
disp_map[disp_map['description'].str.contains('Expired|Hospice', case=False, na=False)]

,id_str,description
21,11,Expired
23,13,Hospice / home
24,14,Hospice / medical facility
29,19,"Expired at home. Medicaid only, hospice."
30,20,"Expired in a medical facility. Medicaid only, ..."
31,21,"Expired, place unknown. Medicaid only, hospice."


In [8]:
expired_ids = [11, 19, 20, 21]
n_expired = df['discharge_disposition_id'].isin(expired_ids).sum()
print(f'Expired encounters: {n_expired} ({round(100*n_expired/len(df),2)}%)')

Expired encounters: 1652 (1.62%)


**Decision:** drop encounters with `discharge_disposition_id` in {11, 19, 20, 21} (Expired) before modeling — this is the standard exclusion used in the original Strack et al. (2014) study on this dataset. Hospice (`13`, `14`) is kept: those patients are alive and technically still at risk of a readmission event, though it's worth flagging in the intervention layer that hospice patients are not realistic care-management targets.

## 4. Non-independence — repeat patients

In [9]:
enc_per_patient = df['patient_nbr'].value_counts()
print('Unique patients:', df['patient_nbr'].nunique())
print('Total encounters:', len(df))
print('Patients with >1 encounter:', (enc_per_patient > 1).sum())
print('Max encounters for a single patient:', enc_per_patient.max())

Unique patients: 71518
Total encounters: 101766
Patients with >1 encounter: 16773
Max encounters for a single patient: 40


**Decision:** 16,773 of 71,518 patients (23%) appear more than once, up to 40 times. Encounters from the same patient are correlated, so:

- Train/test/validation splits must be done by `patient_nbr` (`GroupShuffleSplit` / `GroupKFold`), never by raw encounter row — otherwise the same patient's history leaks across splits and inflates reported performance.
- For the decision-analytics layer (who gets an intervention), the unit of action is the **patient**, not the encounter — so downstream, later encounters likely need to be resolved down to one row per patient (e.g. most recent encounter, or first encounter — to be decided in feature engineering with a stated rationale, not silently).

## 5. Summary of cleaning decisions (implemented in `src/preprocessing.py`)

| Column(s) | Issue | Decision |
|---|---|---|
| `weight` | 97% missing | Drop column |
| `examide`, `citoglipton` | Constant | Drop column |
| `race`, `payer_code`, `medical_specialty`, `diag_1/2/3` | `?` sentinel | Recode to `"Unknown"`, keep as category |
| `max_glu_serum`, `A1Cresult` | `NaN` = not tested | Recode to `"Not tested"` category, never imputed |
| `discharge_disposition_id` in {11,19,20,21} | Expired — cannot be readmitted (leakage) | Drop these encounters |
| `readmitted` | 3-class | Collapse to binary `readmitted_30d` |
| `patient_nbr` | 23% of patients have repeat encounters | Group-aware splitting; resolve to one row/patient downstream |

Next notebook: `02_eda.ipynb` — univariate/bivariate exploration of the cleaned data.